In [1]:
!pip install -q -U datasets tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


In [2]:
import os
import math
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tokenizers import Tokenizer

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", DEVICE)

Using: cuda


In [5]:
SEED = 61

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
INPUT_DIR = "/kaggle/input"

for name in os.listdir(INPUT_DIR):
    print(name)

datasets


In [7]:
BASE_PATH = "/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048"

print(os.listdir(BASE_PATH))

['legal_data', '__huggingface_repos__.json', 'indian_legal_2048', 'indian_legal_tokenizer']


In [8]:
DATASET_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_2048"
)

TOKENIZER_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_tokenizer",
    "tokenizer.json"
)

print(DATASET_PATH)
print(TOKENIZER_PATH)

/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_2048
/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_tokenizer/tokenizer.json


In [9]:
dataset = load_from_disk(DATASET_PATH)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 155060
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 17474
    })
})


In [10]:
tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 16000


In [11]:
PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

PAD: 0
BOS: 2
EOS: 3


In [12]:
example = dataset["train"][0]["input_ids"]

print("Sequence length:", len(example))
print("First 20 IDs:", example[:20])

Sequence length: 2048
First 20 IDs: [2, 5022, 5554, 1102, 5889, 1398, 9, 2556, 100, 606, 83, 4379, 9, 5784, 241, 77, 499, 110, 83, 77]


In [13]:
print(
    tokenizer.decode(example[:500])
)

ĠCIVIL ĠAPPELLATE ĠCivil ĠAppeals ĠNos . 196 Ġto Ġ201 Ġof Ġ1953 . Appeals Ġfrom Ġthe Ġjudgment Ġand Ġof Ġthe ĠPunjab ĠHigh ĠCourt Ġdated Ġ30 , Ġ1949 , Ġin ĠCivil ĠRegular ĠAppeals ĠNos . 15 67 , Ġ15 68 , Ġ15 69 , Ġ15 70 , Ġ15 73 Ġan Ġ15 74 Ġof Ġ1942 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ31 , Ġ1942 , Ġof Ġthe ĠCourt Ġof Ġthe ĠDistrict ĠJudge , ĠHoshiarpur Ġin ĠAppeals ĠNos . 104 / 35 Ġof Ġ1941 - Ġ42 , 101 / 32 Ġof Ġ1941 , Ġ103 / 34 Ġ- of Ġ1941 / 42 ) Ġ15 / 73 Ġof Ġ1941 , Ġ102 / 33 Ġof Ġ1941 / 42 Ġand Ġ120 Ġof Ġ1941 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ24 , Ġ1941 , Ġof Ġthe ĠCourt Ġof ĠSubordinate ĠJudge , Ġ4 th ĠClass , ĠK ang ra Ġin ĠSu its ĠNos . 54 4 , Ġ5 48 , Ġ5 45 , Ġ5 47 , Ġ5 46 Ġand Ġ5 49 Ġof Ġ1940 . B ang ĠBe har ilal Ġand ĠK . ĠR . ĠChaudh ury , Ġfor Ġthe Ġappellant . G an pat ĠRai , Ġfor Ġthe Ġrespondent . M . ĠSik ri , Ġfor ĠPunjab , ĠJ indra ĠLal Ġand ĠR . ĠDhe bar , Ġfor Ġthe Ġ( State Ġof ĠPunjab ). 1956 . O ct ober Ġ23 . The ĠJudgment Ġof Ġthe ĠCourt Ġwas Ġby ĠK . ĠD AS ĠJ 

In [14]:
CONTEXT_LENGTH = 2048

D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048
NUM_LAYERS = 6

DROPOUT = 0.1

print("Context length:", CONTEXT_LENGTH)
print("d_model:", D_MODEL)
print("Heads:", NUM_HEADS)
print("d_head:", D_MODEL // NUM_HEADS)
print("d_ff:", D_FF)
print("Layers:", NUM_LAYERS)

Context length: 2048
d_model: 512
Heads: 8
d_head: 64
d_ff: 2048
Layers: 6


In [15]:
class LegalDataset(Dataset):

    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]["input_ids"]

In [16]:
train_dataset = LegalDataset(
    dataset["train"]
)

# train_dataset = LegalDataset(
#     dataset["train"].select(range(50))
# )

val_dataset = LegalDataset(
    dataset["validation"]
)

# val_dataset = LegalDataset(
#     dataset["validation"].select(range(10))
# )

print("Train sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))

Train sequences: 155060
Validation sequences: 17474


In [17]:
def collate_fn(batch):

    max_length = max(
        len(sequence)
        for sequence in batch
    )

    input_ids = torch.full(
        (len(batch), max_length),
        PAD_ID,
        dtype=torch.long
    )

    for i, sequence in enumerate(batch):

        input_ids[
            i,
            :len(sequence)
        ] = torch.tensor(
            sequence,
            dtype=torch.long
        )

    return input_ids

In [18]:
BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [19]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len
    ):
        super().__init__()

        position = torch.arange(
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) *
            (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        return x + self.pe[
            :, :x.size(1)
        ]

In [20]:
class TransformerLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        max_len,
        dropout
    ):
        super().__init__()

        self.d_model = d_model

        # Token embedding
        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Positional encoding
        self.position = PositionalEncoding(
            d_model,
            max_len
        )

        # Built-in Transformer layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=False
        )

        # Stack of Transformer layers
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        # Vocabulary projection
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Weight tying
        self.lm_head.weight = self.embedding.weight

    def forward(
        self,
        input_ids,
        padding_mask=None
    ):

        x = self.embedding(
            input_ids
        )

        # Same sqrt(d_model) scaling used in
        # the original Transformer
        x = x * math.sqrt(
            self.d_model
        )

        x = self.position(x)

        seq_len = input_ids.size(1)

        # Causal mask:
        # token cannot see future tokens
        causal_mask = (
            torch.triu(
                torch.ones(
                    seq_len,
                    seq_len,
                    device=input_ids.device
                ),
                diagonal=1
            ).bool()
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)

        logits = self.lm_head(x)

        return logits

In [21]:
model = TransformerLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    max_len=CONTEXT_LENGTH,
    dropout=DROPOUT
)

model = model.to(DEVICE)

In [22]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

Total parameters: 27,107,328
Trainable parameters: 27,107,328


In [23]:
batch = next(
    iter(train_loader)
)

batch = batch.to(
    DEVICE
)

print("Input:", batch.shape)

padding_mask = (
    batch == PAD_ID
)

with torch.no_grad():

    logits = model(
        batch,
        padding_mask
    )

print("Output:", logits.shape)

Input: torch.Size([1, 1191])
Output: torch.Size([1, 1191, 16000])


In [24]:
test_length = 8

mask = torch.triu(
    torch.ones(
        test_length,
        test_length
    ),
    diagonal=1
).bool()

print(
    (~mask).int()
)

tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)


In [25]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

In [26]:
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [27]:
EPOCHS = 1

GRAD_ACCUMULATION_STEPS = 8

In [28]:
TOTAL_STEPS = math.ceil(
    len(train_loader) / GRAD_ACCUMULATION_STEPS
)

WARMUP_STEPS = int(
    0.1 * TOTAL_STEPS
)

print("Total optimizer steps:", TOTAL_STEPS)
print("Warmup steps:", WARMUP_STEPS)

Total optimizer steps: 19383
Warmup steps: 1938


In [29]:
def lr_lambda(current_step):

    if current_step < WARMUP_STEPS:

        return (
            current_step /
            max(1, WARMUP_STEPS)
        )

    return max(
        0.0,

        (
            TOTAL_STEPS - current_step
        )
        /
        max(
            1,
            TOTAL_STEPS - WARMUP_STEPS
        )
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

In [30]:
def train_one_epoch():
    model.train()
    total_loss = 0.0
    
    os.makedirs(
        "/kaggle/working/indian_legal_transformer",
        exist_ok=True
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    optimizer.zero_grad(set_to_none=True)

    # ── Fix 4: track best loss ──────────────────────────────
    best_loss = float('inf')
    # ────────────────────────────────────────────────────────

    for step, batch in enumerate(train_loader):
        batch = batch.to(DEVICE, non_blocking=True)
        inputs = batch[:, :-1]
        labels = batch[:, 1:]
        padding_mask = (inputs == PAD_ID)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            logits = model(inputs, padding_mask)
            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                labels.reshape(-1)
            )
            loss = loss / GRAD_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)

            # ── Fix 2: capture grad_norm instead of discarding it ──
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # ──────────────────────────────────────────────────────

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            # ── Fix 4: save checkpoint whenever loss improves ──────
            current_loss = loss.item() * GRAD_ACCUMULATION_STEPS
            if current_loss < best_loss:
                best_loss = current_loss
                torch.save(
                    model.state_dict(),
                    "/kaggle/working/indian_legal_transformer/best_checkpoint.pt"
                )
            # ──────────────────────────────────────────────────────

        total_loss += loss.item()

        # ── Fix 3: log grad_norm alongside loss and LR ────────────
        if (step + 1) % 100 == 0:
            print(
                f"Step {step + 1:,} | "
                f"Loss: {loss.item() * GRAD_ACCUMULATION_STEPS:.4f} | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                f"Grad norm: {grad_norm:.4f}"
            )
        # ──────────────────────────────────────────────────────────

    return (total_loss / len(train_loader)) * GRAD_ACCUMULATION_STEPS

In [31]:
@torch.no_grad()
def evaluate():

    model.eval()

    total_loss = 0.0

    for batch in val_loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        inputs = batch[:, :-1]

        labels = batch[:, 1:]

        padding_mask = (
            inputs == PAD_ID
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(
                inputs,
                padding_mask
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                labels.reshape(-1)
            )

        total_loss += loss.item()

    return (
        total_loss /
        len(val_loader)
    )

In [32]:
for epoch in range(EPOCHS):

    print(
        f"\n{'='*50}"
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"{'='*50}"
    )

    train_loss = train_one_epoch()

    val_loss = evaluate()

    train_ppl = math.exp(
        min(train_loss, 20)
    )

    val_ppl = math.exp(
        min(val_loss, 20)
    )

    print(
        f"\nTrain loss: {train_loss:.4f}"
    )

    print(
        f"Validation loss: {val_loss:.4f}"
    )

    print(
        f"Train perplexity: {train_ppl:.2f}"
    )

    print(
        f"Validation perplexity: {val_ppl:.2f}"
    )


Epoch 1/1
Step 100 | Loss: 195.1154 | LR: 6.19e-07 | Grad norm: 382.1346
Step 200 | Loss: 179.0404 | LR: 1.29e-06 | Grad norm: 387.8211
Step 300 | Loss: 165.1641 | LR: 1.91e-06 | Grad norm: 394.0605
Step 400 | Loss: 146.1816 | LR: 2.58e-06 | Grad norm: 381.8535
Step 500 | Loss: 114.7810 | LR: 3.20e-06 | Grad norm: 307.8922
Step 600 | Loss: 92.8466 | LR: 3.87e-06 | Grad norm: 162.8226
Step 700 | Loss: 77.3783 | LR: 4.49e-06 | Grad norm: 100.6105
Step 800 | Loss: 65.9593 | LR: 5.16e-06 | Grad norm: 68.8879
Step 900 | Loss: 57.3727 | LR: 5.78e-06 | Grad norm: 53.0197
Step 1,000 | Loss: 52.3293 | LR: 6.45e-06 | Grad norm: 37.8273
Step 1,100 | Loss: 46.3128 | LR: 7.07e-06 | Grad norm: 29.8769
Step 1,200 | Loss: 45.4618 | LR: 7.74e-06 | Grad norm: 22.5163
Step 1,300 | Loss: 39.2252 | LR: 8.36e-06 | Grad norm: 20.5443
Step 1,400 | Loss: 40.9215 | LR: 9.03e-06 | Grad norm: 18.1498
Step 1,500 | Loss: 36.7637 | LR: 9.65e-06 | Grad norm: 18.2979
Step 1,600 | Loss: 33.6455 | LR: 1.03e-05 | Grad n

In [33]:
MODEL_DIR = "/kaggle/working/indian_legal_transformer"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

torch.save(
    model.state_dict(),
    os.path.join(
        MODEL_DIR,
        "transformer_model.pt"
    )
)

In [34]:
config = {
    "architecture": "Transformer Causal Language Model",
    "vocab_size": VOCAB_SIZE,
    "context_length": CONTEXT_LENGTH,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "d_head": D_MODEL // NUM_HEADS,
    "d_ff": D_FF,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED
}

with open(
    os.path.join(
        MODEL_DIR,
        "config.json"
    ),
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print(
    "Model saved to:",
    MODEL_DIR
)

Model saved to: /kaggle/working/indian_legal_transformer


In [35]:
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=100,
    temperature=0.8
):

    model.eval()

    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):

        input_ids = input_ids[
            :, -CONTEXT_LENGTH:
        ]

        padding_mask = (
            input_ids == PAD_ID
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(
                input_ids,
                padding_mask
            )

        next_token_logits = (
            logits[:, -1, :]
            / temperature
        )

        probabilities = F.softmax(
            next_token_logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        input_ids = torch.cat(
            [
                input_ids,
                next_token
            ],
            dim=1
        )

        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(
        input_ids[0].tolist()
    )

In [36]:
prompt = "The Supreme Court of India"

generated = generate(
    prompt,
    max_new_tokens=100,
    temperature=0.8
)

print(generated)

ĠThe ĠSupreme ĠCourt Ġof ĠIndia Ġand Ġpractice Ġunder Ġs . 64 , ĠJ . Ġconsult Ġsworn , Ġas Ġpremises Ġin Ġterms Ġof ĠPolice , Ġthey Ġdo Ġso Ġfar Ġas Ġhis Ġreport , Ġ1994 Ġat Ġthe Ġappellant , Ġ5 Ġ( Ġdeposited Ġby Ġthe Ġand Ġin ĠIndia . It Ġwas Ġthe Ġappellant . The Ġappellant Ġand ĠDelhi Ġfor Ġthe Ġappellant Ġand Ġthe Ġappellant Ġfirst Ġtime Ġof Ġthe Ġappellant , Ġand Ġthe ĠSupreme ĠCourt . This Ġdecision Ġof Ġthe Ġappellant Ġwas Ġexamined Ġin Ġthe Ġappellant Ġhad Ġgiven Ġsubject Ġmatter Ġof Ġthe Ġsuit Ġwas Ġalso Ġsubmitted Ġthat Ġhe Ġhad Ġbeen Ġgiven Ġby Ġthe Ġappellant Ġsubmitted Ġthat Ġhe Ġwas Ġa
